# 05_rb_ate.ipynb
## Purpose
RB extraction of noun phrases / candidates using POS + dependency patterns.

## Expected inputs
- `data/segments_index.csv`
- `preprocessed RB text`

## Expected outputs
- `outputs/rb_output.csv`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

# 1) Setup

In [ ]:

# ============================================================
# 1) Setup
# ============================================================
# Install (sekali saja jika belum ada):
# !pip -q install pandas tqdm stanza==1.8.2

import re, json, time, platform
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

import stanza

print("Python:", platform.python_version())

# 2) Config

In [ ]:

# ============================================================
# 2) Config
# ============================================================
@dataclass
class Config:
    input_csv: str = "data\\preprocessedfor_rb_with_meta.csv"
    text_col: str = "seg_text_processed"
    out_dir: str = "data\\outputs_rb"
    join_cols_prefer: tuple = ("comment_id","comment_ori","seg_id","source","date")

    stanza_lang_primary: str = "id"
    stanza_lang_fallback: str = "en"

    interjeksi_file: str = "data\\interjeksi_stopwords.txt"

    use_amod_in_phrase: bool = True
    allow_single_noun_rules: bool = True

cfg = Config()
print(cfg)

# 3) Load data

In [ ]:

# ============================================================
# 3) Load data
# ============================================================
INP = Path(cfg.input_csv)
assert INP.exists(), f"File tidak ditemukan: {INP.resolve()}"

df = pd.read_csv(INP)
assert cfg.text_col in df.columns, f"Kolom '{cfg.text_col}' tidak ada. Kolom tersedia: {list(df.columns)}"

df[cfg.text_col] = df[cfg.text_col].fillna("").astype(str)
df = df[df[cfg.text_col].str.strip() != ""].copy()

join_cols = [c for c in cfg.join_cols_prefer if c in df.columns]
print("Rows:", len(df))
print("Join cols:", join_cols)

df[[cfg.text_col] + join_cols].head(3)

# 4) Load interjection list + normalizer

In [ ]:

# ============================================================
# 4) Load interjection list + normalizer
# ============================================================
def load_wordset(path: str):
    p = Path(path)
    assert p.exists(), f"File interjeksi tidak ditemukan: {p.resolve()}"
    return {w.strip().lower() for w in p.read_text(encoding="utf-8", errors="ignore").splitlines() if w.strip()}

INTERJ = load_wordset(cfg.interjeksi_file)
print("Interjection count:", len(INTERJ))

_punct_strip = re.compile(r"(^\W+|\W+$)", flags=re.UNICODE)

def normalize_token(tok: str) -> str:
    tok = str(tok).lower().strip()
    tok = _punct_strip.sub("", tok)  # "ckckck." -> "ckckck", "hah?" -> "hah"
    return tok

def remove_interjections(text: str):
    toks = str(text).split()
    kept, removed = [], []
    for tok in toks:
        nt = normalize_token(tok)
        if nt in INTERJ:
            removed.append(nt)
        else:
            kept.append(tok)
    return " ".join(kept).strip(), removed

df["seg_text_clean"], df["interj_removed"] = zip(*df[cfg.text_col].progress_apply(remove_interjections))
df["n_interj_removed"] = df["interj_removed"].apply(len)

print("Rows with interjection removed:", int((df["n_interj_removed"]>0).sum()))
df[[cfg.text_col, "seg_text_clean", "interj_removed"]].head(5)

# 5) Stanza pipeline

In [ ]:
# ============================================================
# 5) Stanza pipeline (Deps BUT lemma required by Stanza)
# ============================================================

import stanza

# Jalankan SEKALI (if belum pernah download modelnya)
# Kalau sudah pernah, ini will skip/cepat.
stanza.download("id")
stanza.download("en")

def build_pipeline(lang: str):
    return stanza.Pipeline(
        lang=lang,
        processors="tokenize,pos,lemma,depparse",  # <-- tambahkan lemma
        tokenize_no_ssplit=True,
        verbose=False
    )

try:
    nlp_id = build_pipeline(cfg.stanza_lang_primary)
except Exception as e:
    print("Gagal init stanza ID:", e)
    nlp_id = None

try:
    nlp_en = build_pipeline(cfg.stanza_lang_fallback)
except Exception as e:
    print("Gagal init stanza EN:", e)
    nlp_en = None

assert (nlp_id is not None) or (nlp_en is not None), \
    "Stanza pipeline gagal. Pastikan stanza.download('id')/('en') sukses dan kernel restart."

# 6) Rule-based extraction 

In [ ]:
# ============================================================
# 6) Rule-based extraction (v2) — list output, safer rules
# ============================================================
def extract_aspect_terms_v2(text: str, nlp):
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)
    out, seen = [], set()

    for sent in doc.sentences:
        words = sent.words
        by_id = {w.id: w for w in words}

        allowed = {"compound", "flat"}
        if cfg.use_amod_in_phrase:
            allowed.add("amod")

        # (1) Phrase rule: head NOUN/PROPN + (compound/flat/amod children)
        for head in words:
            if head.upos not in ("NOUN", "PROPN"):
                continue
            children = [w for w in words if (w.head == head.id and w.deprel in allowed)]
            phrase_words = sorted([head] + children, key=lambda x: x.id)
            phrase = " ".join(w.text for w in phrase_words).strip()
            key = phrase.lower()
            if key and key not in seen:
                out.append(phrase)
                seen.add(key)

        # (2) Single noun selective rule
        if cfg.allow_single_noun_rules:
            good_deps = {"nsubj","nsubj:pass","obj","obl","nmod","iobj"}
            for w in words:
                if w.upos not in ("NOUN","PROPN"):
                    continue
                if w.deprel not in good_deps:
                    continue
                if w.head <= 0:
                    continue
                head = by_id.get(w.head)
                if head is None:
                    continue
                if head.upos in ("VERB","ADJ"):
                    key = w.text.lower()
                    if key and key not in seen:
                        out.append(w.text)
                        seen.add(key)

    return out

def filter_terms_interj(terms):
    kept, removed = [], []
    for t in terms:
        nt = normalize_token(t)
        if not nt:
            continue
        if nt in INTERJ:
            removed.append(nt)
        else:
            kept.append(t)
    return kept, removed

def run_extract(text: str):
    terms_id = extract_aspect_terms_v2(text, nlp_id) if nlp_id is not None else []
    if terms_id:
        return terms_id, "id"
    terms_en = extract_aspect_terms_v2(text, nlp_en) if nlp_en is not None else []
    if terms_en:
        return terms_en, "en"
    return [], "none"

# 7) Apply extraction + post-filter

In [ ]:

# ============================================================
# 7) Apply extraction + post-filter
# ============================================================
t0 = time.time()

raw_terms, langs = [], []
for txt in tqdm(df["seg_text_clean"].tolist(), desc="Extracting RB aspects"):
    t, lang = run_extract(txt)
    raw_terms.append(t)
    langs.append(lang)

df["aspect_terms_rb_raw"] = raw_terms
df["rb_lang_used"] = langs

kept_list, removed_list = [], []
for terms in tqdm(df["aspect_terms_rb_raw"].tolist(), desc="Post-filter interjections"):
    kept, rem = filter_terms_interj(terms)
    kept_list.append(kept)
    removed_list.append(rem)

df["aspect_terms_rb"] = kept_list
df["rb_removed_terms"] = removed_list

df["n_aspect_raw"] = df["aspect_terms_rb_raw"].apply(len)
df["n_aspect_final"] = df["aspect_terms_rb"].apply(len)

t1 = time.time()
print("Elapsed sec:", round(t1-t0, 2))

df[["seg_text_clean","aspect_terms_rb_raw","aspect_terms_rb"]].head(5)

## 7b) Post-filter stopwords (ID+EN+custom)

In [ ]:
import re, ast
import pandas as pd

# lebih kuat for punctuation unicode
EDGE_PUNCT = re.compile(r"(^[^\w]+|[^\w]+$)", flags=re.UNICODE)
PUNCT_STRIP = ".,!?;:()[]{}\"'`…—–"

def norm_token(tok: str) -> str:
    tok = "" if tok is None else str(tok)
    tok = tok.strip().lower()
    tok = tok.strip(PUNCT_STRIP)
    tok = EDGE_PUNCT.sub("", tok)
    return tok

def ensure_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        t = x.strip()
        if t.startswith("[") and t.endswith("]"):
            try:
                v = ast.literal_eval(t)
                return v if isinstance(v, list) else [v]
            except Exception:
                return [t]
        return [t]
    return [x]

def strip_stopwords_in_phrase(phrase: str, stopset: set) -> str:
    """Hapus token stopword di dalam phrase, case-insensitive, dan buang punct di tepi token."""
    kept_tokens = []
    for tok in str(phrase).split():
        core = norm_token(tok)
        if not core:
            continue
        if core in stopset:
            continue
        kept_tokens.append(core)
    return " ".join(kept_tokens).strip()

def post_filter_terms_aggressive(terms, stopwords: set):
    """
    - Buang term jika setelah remove-stopwords menjadi kosong
    - Buang term jika hasil akhirnya 1 kata dan itu stopword
    - Normalisasi output ke lowercase + token core
    - Unik & preserve order
    """
    terms = ensure_list(terms)
    kept, removed = [], []

    for term in terms:
        raw = "" if term is None else str(term)
        cleaned = strip_stopwords_in_phrase(raw, stopwords)

        # if habis dibersihkan jadi empty -> remove
        if not cleaned:
            removed.append(norm_token(raw) or raw)
            continue

        # safety: if resultsnya still stopword tunggal -> remove
        if cleaned in stopwords:
            removed.append(cleaned)
            continue

        kept.append(cleaned)

    # unik preserve order
    seen, kept_u = set(), []
    for t in kept:
        if t not in seen:
            seen.add(t)
            kept_u.append(t)

    return kept_u, removed

# ===
# --- Load custom stopwords
def load_stopwords(path: str) -> set:
    sw = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t = norm_token(line)
            if t:
                sw.add(t)
    return sw

CUSTOM = load_stopwords("data//custom_stopwords.txt")  # sesuaikan path jika perlu

# --- Stopwords EN/ID
STOP_ID, STOP_EN = set(), set()
try:
    import nltk
    from nltk.corpus import stopwords
    try:
        STOP_ID = set(stopwords.words("indonesian"))
    except Exception:
        nltk.download("stopwords")
        STOP_ID = set(stopwords.words("indonesian"))
    STOP_EN = set(stopwords.words("english"))
except Exception:
    try:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        STOP_EN = set(ENGLISH_STOP_WORDS)
    except Exception:
        STOP_EN = set()
    STOP_ID = {
        "yang","dan","di","ke","dari","ini","itu","atau","pada","untuk","dengan","dalam","sebagai",
        "adalah","ialah","nya","lah","kah","pun","juga","saja","jadi","karena","agar","bukan","tidak",
        "ya","kok","sih","dong","deh","nih","tuh","mah","iya"
    }

STOP_ID = {norm_token(x) for x in STOP_ID if norm_token(x)}
STOP_EN = {norm_token(x) for x in STOP_EN if norm_token(x)}

# --- Extra filler yang sering lolos jadi "aspek"
EXTRA = {
    "dll","dst","dsb","etc","cmiiw","imo","imho","hi"
    "wkwk","wk","haha","hehe","hihi","ckck","ckckck","hah","ah","eh","heh","hm","hmm","uh",
    "btw","fyi","lol","lmao","rofl",
}
EXTRA = {norm_token(x) for x in EXTRA if norm_token(x)}

STOPWORDS = set().union(CUSTOM, STOP_ID, STOP_EN, EXTRA)

# ===

# ==== apply to dataframe ====
tmp = df["aspect_terms_rb"].apply(lambda x: post_filter_terms_aggressive(x, STOPWORDS))
df["aspect_terms_rb_clean"] = tmp.apply(lambda z: z[0])
df["rb_removed_stopwords"]  = tmp.apply(lambda z: z[1])

df[["seg_text_processed","aspect_terms_rb","aspect_terms_rb_clean","rb_removed_stopwords"]].head(15)

# 8) Save outputs + audit

In [ ]:

# ============================================================
# 8) Save outputs + audit
# ============================================================
OUT = Path(cfg.out_dir)
OUT.mkdir(parents=True, exist_ok=True)

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# row-level audit
audit_row = pd.DataFrame({
    "row_id": np.arange(len(df)),
    "n_interj_removed": df["n_interj_removed"].astype(int),
    "interj_removed": df["interj_removed"],
    "rb_lang_used": df["rb_lang_used"],
    "n_aspect_raw": df["n_aspect_raw"].astype(int),
    "n_aspect_final": df["n_aspect_final"].astype(int),
    "rb_removed_terms": df["rb_removed_terms"],
})
for c in join_cols:
    audit_row[c] = df[c]

audit_row.to_csv(OUT / "rb_audit_rowlevel.csv", index=False, encoding="utf-8")

# full output

cols_full = join_cols + [cfg.text_col, "seg_text_clean", "interj_removed", "aspect_terms_rb_raw", "aspect_terms_rb","aspect_terms_rb_clean","rb_lang_used"]
df[cols_full].to_csv(OUT / "rb_aspect_terms_full.csv", index=False, encoding="utf-8")

# compact output
cols_compact = join_cols + ["aspect_terms_rb"] if join_cols else ["aspect_terms_rb"]
df[cols_compact].to_csv(OUT / "rb_aspect_terms_compact.csv", index=False, encoding="utf-8")

# summary audit json
audit = {
    "created_at": datetime.now().isoformat(),
    "config": asdict(cfg),
    "dataset": {"input_csv": str(INP), "n_rows": int(len(df)), "join_cols": join_cols},
    "interjection": {
        "interjeksi_file": cfg.interjeksi_file,
        "interjeksi_count": int(len(INTERJ)),
        "rows_with_removed": int((df["n_interj_removed"]>0).sum()),
        "avg_removed_per_row": float(df["n_interj_removed"].mean()),
    },
    "aspect_terms": {
        "avg_raw": float(df["n_aspect_raw"].mean()),
        "avg_final": float(df["n_aspect_final"].mean()),
        "rows_empty_final": int((df["n_aspect_final"]==0).sum()),
        "lang_used_counts": df["rb_lang_used"].value_counts().to_dict(),
    },
    "stopwords_postfilter": {
    "stopwords_size": int(len(STOPWORDS)),
    "n_rows_removed_any": int((df["rb_removed_stopwords"].apply(len) > 0).sum()),
    "removed_total_terms": int(df["rb_removed_stopwords"].apply(len).sum()),
    "top_removed_terms": (pd.Series([t for xs in df["rb_removed_stopwords"] for t in xs])
                          .value_counts().head(20).to_dict())
    }
}


save_json(audit, OUT / "rb_audit.json")

print("Saved:")
print(" -", (OUT / "rb_aspect_terms_full.csv").resolve())
print(" -", (OUT / "rb_aspect_terms_compact.csv").resolve())
print(" -", (OUT / "rb_audit_rowlevel.csv").resolve())
print(" -", (OUT / "rb_audit.json").resolve())